# Análisis Avanzado de Series Temporales
## Modelos ARIMA y SARIMA para Horizon Digital
Este notebook explora técnicas avanzadas de análisis de series temporales, incluyendo modelos ARIMA y SARIMA, para identificar patrones y realizar pronósticos basados en datos históricos.

In [ ]:
# Carga de bibliotecas y datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset
df = pd.read_csv('ecommerce_data_1M.csv')
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'])
df.set_index('LastPurchaseDate', inplace=True)

# Crear la serie temporal de ventas mensuales
monthly_sales = df['LastPurchaseAmount'].resample('M').sum()
monthly_sales.plot(figsize=(10,5), title='Ventas Mensuales Históricas')
plt.ylabel('Ventas (USD)')
plt.show()

In [ ]:
# Análisis exploratorio de la serie temporal
monthly_sales.plot(figsize=(10,5), title='Tendencia de Ventas Mensuales')
plt.xlabel('Fecha')
plt.ylabel('Ventas (USD)')
plt.show()

# Descomposición de la serie temporal
result = seasonal_decompose(monthly_sales, model='additive', period=12)
result.plot()
plt.show()

In [ ]:
# Modelado con ARIMA
model_arima = ARIMA(monthly_sales, order=(1,1,1))
arima_fit = model_arima.fit()

# Evaluación del modelo ARIMA
arima_pred = arima_fit.predict(start=len(monthly_sales), end=len(monthly_sales)+11, typ='levels')
plt.figure(figsize=(10,5))
plt.plot(monthly_sales, label='Histórico')
plt.plot(arima_pred, label='Pronóstico ARIMA', color='red')
plt.title('Pronóstico con ARIMA')
plt.legend()
plt.show()

In [ ]:
# Modelado con SARIMA
model_sarima = SARIMAX(monthly_sales, order=(1,1,1), seasonal_order=(1,1,1,12))
sarima_fit = model_sarima.fit(disp=False)

# Evaluación del modelo SARIMA
sarima_pred = sarima_fit.get_forecast(steps=12)
sarima_mean = sarima_pred.predicted_mean
sarima_conf_int = sarima_pred.conf_int()

plt.figure(figsize=(10,5))
plt.plot(monthly_sales, label='Histórico')
plt.plot(sarima_mean, label='Pronóstico SARIMA', color='green')
plt.fill_between(sarima_mean.index, sarima_conf_int.iloc[:,0], sarima_conf_int.iloc[:,1], color='lightgreen', alpha=0.3)
plt.title('Pronóstico con SARIMA')
plt.legend()
plt.show()

In [ ]:
# Comparación de Modelos y Conclusiones
rmse_arima = mean_squared_error(monthly_sales[-12:], arima_pred[:12], squared=False)
rmse_sarima = mean_squared_error(monthly_sales[-12:], sarima_mean[:12], squared=False)

print(f'RMSE ARIMA: {rmse_arima:.2f}')
print(f'RMSE SARIMA: {rmse_sarima:.2f}')

# Conclusión
if rmse_arima < rmse_sarima:
    print('El modelo ARIMA tiene un mejor rendimiento basado en RMSE.')
else:
    print('El modelo SARIMA tiene un mejor rendimiento basado en RMSE.')